# SWAPI → Open Mirroring with Microsoft Fabric (Clean Initial Load)

This notebook performs a **clean initial load** into an **Open Mirrored Database**
in Microsoft Fabric using the public **Star Wars API (SWAPI)**.

It will:

- Use the **Landing zone URL** from an existing Open Mirrored Database (Generic mirror).
- Write Open Mirroring–compatible **Parquet files** into the Landing Zone.
- Create one mirrored table per SWAPI endpoint:
  - `sw_people`
  - `sw_planets`
  - `sw_films`
  - `sw_species`
  - `sw_starships`
  - `sw_vehicles`

Design decisions for **initial load**:

- **No `__rowMarker__` column** in Parquet (Open Mirroring treats all rows as INSERTs).
- Per-table `_metadata.json` only contains:
  ```json
  { "keyColumns": ["<key_column>"] }
  ```
- One Parquet file per table, named `00000000000000000001.parquet`.

---
## Prerequisites

1. You have created an **Open Mirrored Database (Generic mirror)** in Fabric.
2. From that Mirrored DB, copy the **Landing zone URL** from the Home page.
3. Paste it into the configuration cell below as `landing_zone_https`.


In [ ]:
# STEP 1 – CONFIGURATION
# ----------------------
# Paste the Landing zone URL from the Mirrored Database home page.
# Example:
# landing_zone_https = "https://onelake.dfs.fabric.microsoft.com/<workspace-id>/<mirrored-db-id>/Files/LandingZone"

landing_zone_https = "[Landing Zone URL]"  # TODO: replace with your URL

from urllib.parse import urlparse

def https_to_abfss(url: str) -> str:
    """Convert a OneLake HTTPS URL to an abfss path usable by mssparkutils.fs.

    Example:
      https://onelake.dfs.fabric.microsoft.com/ws-id/item-id/Files/LandingZone
        -> abfss://ws-id@onelake.dfs.fabric.microsoft.com/item-id/Files/LandingZone
    """
    parsed = urlparse(url)
    if parsed.scheme != "https" or "onelake.dfs.fabric.microsoft.com" not in parsed.netloc:
        raise ValueError(f"URL doesn't look like a OneLake landing zone URL: {url}")

    parts = [p for p in parsed.path.split("/") if p]
    if len(parts) < 3:
        raise ValueError(f"Unexpected OneLake path structure: {parsed.path}")

    workspace_id = parts[0]
    item_id = parts[1]
    subpath = "/".join(parts[2:])

    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{item_id}/{subpath}"

# Root of the Landing Zone as an abfss URI
landing_zone_root = https_to_abfss(landing_zone_https.rstrip("/"))

# Map SWAPI endpoints to mirrored table names and key column names
endpoints_config = {
    "people":   {"table_name": "sw_people",   "key_column": "person_id"},
    "planets":  {"table_name": "sw_planets",  "key_column": "planet_id"},
    "films":    {"table_name": "sw_films",    "key_column": "film_id"},
    "species":  {"table_name": "sw_species",  "key_column": "species_id"},
    "starships":{"table_name": "sw_starships","key_column": "starship_id"},
    "vehicles": {"table_name": "sw_vehicles", "key_column": "vehicle_id"},
}

print("Landing zone (HTTPS):", landing_zone_https)
print("Landing zone (abfss):", landing_zone_root)
print("Configured endpoints → tables:")
for ep, cfg in endpoints_config.items():
    print(f"  - {ep} -> {cfg['table_name']} (key: {cfg['key_column']})")

In [ ]:
# STEP 2 – DEPENDENCIES & HELPERS
# --------------------------------

# If requests is not already installed in this environment, uncomment:
# %pip install requests

import json
import re
import requests

from notebookutils import mssparkutils
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType
)

def fetch_swapi_collection(endpoint: str):
    """Fetch all pages for a given SWAPI endpoint (e.g., 'people', 'planets')."""
    url = f"https://swapi.dev/api/{endpoint}/"
    results = []
    while url:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        results.extend(data.get("results", []))
        url = data.get("next")
    return results

def extract_id_from_url(url: str) -> int:
    """Extract numeric ID from SWAPI resource URL, e.g. '.../people/1/'."""
    return int(url.rstrip("/").split("/")[-1])

def normalize_record(rec: dict) -> dict:
    """Convert lists/dicts to JSON strings so we can store everything as strings."""
    out = {}
    for k, v in rec.items():
        if isinstance(v, (list, dict)):
            out[k] = json.dumps(v)
        else:
            out[k] = v
    return out

def get_next_sequence_number(folder: str) -> int:
    """Return the next integer sequence based on existing 20-digit parquet filenames."""
    try:
        files = mssparkutils.fs.ls(folder)
    except Exception:
        return 1

    seq_nums = []
    for f in files:
        name = f.name
        if name.endswith(".parquet") and re.match(r"^\d{20}\.parquet$", name):
            seq_nums.append(int(name.split(".")[0]))

    return max(seq_nums) + 1 if seq_nums else 1

def get_sequence_filename(n: int) -> str:
    """Format integer as a 20-digit zero-padded filename."""
    return f"{n:020d}.parquet"

def ensure_metadata(table_folder_path: str, key_column: str):
    """Ensure _metadata.json exists for this table, with key_column as keyColumns."""
    mssparkutils.fs.mkdirs(table_folder_path)
    metadata_path = f"{table_folder_path}/_metadata.json"

    try:
        mssparkutils.fs.head(metadata_path, 1)
        print("_metadata.json already exists:", metadata_path)
        return
    except Exception:
        pass

    metadata = {
        "keyColumns": [key_column]
    }

    mssparkutils.fs.put(
        metadata_path,
        json.dumps(metadata, indent=2),
        overwrite=True
    )
    print("Created _metadata.json at", metadata_path)

def build_dataframe_for_endpoint(endpoint: str, table_name: str, key_column: str, records: list):
    """Build a Spark DataFrame for a SWAPI endpoint for INITIAL LOAD.

    - **No** __rowMarker__ column (initial load = full inserts).
    - Adds an integer key column derived from the trailing ID in the 'url' field.
    - All other fields are stored as strings.
    """
    if not records:
        raise ValueError(f"No records returned for endpoint '{endpoint}'")

    normalized = [normalize_record(r) for r in records]

    # Collect all keys seen across records
    all_keys = set()
    for r in normalized:
        all_keys.update(r.keys())

    if "url" not in all_keys:
        raise ValueError(f"Endpoint '{endpoint}' records do not contain a 'url' field")

    data_columns = sorted(all_keys)  # includes 'url'

    rows = []
    for rec in normalized:
        url = rec.get("url")
        if not url:
            continue

        entity_id = extract_id_from_url(url)

        row_dict = {
            key_column: entity_id,
        }

        for col in data_columns:
            value = rec.get(col)
            row_dict[col] = None if value is None else str(value)

        rows.append(Row(**row_dict))

    # Schema: key_column (int), then all SWAPI fields as string
    schema_fields = [
        StructField(key_column, IntegerType(), nullable=False),
    ]
    for col in data_columns:
        schema_fields.append(StructField(col, StringType(), nullable=True))

    schema = StructType(schema_fields)
    df = spark.createDataFrame(rows, schema=schema)
    return df

print("✅ Helpers loaded. Next: run the 'INITIAL LOAD – ALL ENDPOINTS' cell.")

In [ ]:
# STEP 3 – INITIAL LOAD – ALL SWAPI ENDPOINTS
# -------------------------------------------

for endpoint, cfg in endpoints_config.items():
    table_name = cfg["table_name"]
    key_column = cfg["key_column"]
    table_folder = f"{landing_zone_root}/{table_name}"

    print("\n==============================")
    print(f"Endpoint: {endpoint}")
    print(f"Target table: {table_name} (key: {key_column})")
    print(f"Table folder: {table_folder}")
    print("Fetching from SWAPI...")

    records = fetch_swapi_collection(endpoint)
    print(f"  → {len(records)} records fetched")

    # Ensure metadata for this table
    ensure_metadata(table_folder, key_column)

    # Build DataFrame (NO __rowMarker__ column for initial load)
    df = build_dataframe_for_endpoint(endpoint, table_name, key_column, records)
    print("  → DataFrame created with", df.count(), "rows")

    # Optionally preview for the first endpoint only
    if endpoint == list(endpoints_config.keys())[0]:
        display(df.limit(5))

    # Write a single parquet file to a tmp folder, then rename to 20-digit sequence.
    tmp_path = f"{table_folder}/tmp_initial"

    df.coalesce(1).write.mode("overwrite").parquet(tmp_path)

    files = mssparkutils.fs.ls(tmp_path)
    parquet_files = [f for f in files if f.name.endswith(".parquet")]
    if not parquet_files:
        raise Exception(f"No parquet file written in {tmp_path} for endpoint '{endpoint}'")

    source_file = parquet_files[0].path

    seq_num = get_next_sequence_number(table_folder)
    target_name = get_sequence_filename(seq_num)
    target_path = f"{table_folder}/{target_name}"

    print("  → Moving", source_file, "->", target_path)
    mssparkutils.fs.mv(source_file, target_path)
    mssparkutils.fs.rm(tmp_path, True)

    print("  → Initial parquet created for endpoint:", endpoint)

print("\n✅ Initial load complete for all endpoints.")
print("Open Mirroring will treat these files as INSERTs since there is no __rowMarker__ column.")

## STEP 4 – Validate files in the Landing Zone (optional)

Use the cells below to inspect the Parquet files you just wrote.
Open Mirroring will handle ingestion into the Mirrored Database tables.

In [ ]:
# List table folders and their files
print("Landing zone root (abfss):", landing_zone_root)
print("\nLanding zone contents:")
display(mssparkutils.fs.ls(landing_zone_root))

for endpoint, cfg in endpoints_config.items():
    table_name = cfg["table_name"]
    table_folder = f"{landing_zone_root}/{table_name}"
    print(f"\nFiles for table '{table_name}' in {table_folder}")
    try:
        display(mssparkutils.fs.ls(table_folder))
    except Exception as e:
        print("  (Error listing folder)", e)

In [ ]:
# Read back a sample parquet to verify data (debug/demo only)
sample_table = "sw_people"  # change to sw_planets, sw_films, etc. if desired
sample_path = f"{landing_zone_root}/{sample_table}/00000000000000000001.parquet"

print("Reading sample Parquet:", sample_path)
df_sample = spark.read.parquet(sample_path)
df_sample.printSchema()
print("Row count:", df_sample.count())
display(df_sample.limit(10))

## STEP 5 – Query from the Mirrored SQL Endpoint

Once Open Mirroring has processed these files, you should see the tables and data
in the Mirrored Database **SQL analytics endpoint**.

Expected table names:
- `sw_people`
- `sw_planets`
- `sw_films`
- `sw_species`
- `sw_starships`
- `sw_vehicles`

Example queries:

```sql
SELECT TOP 10 * FROM dbo.sw_people;
SELECT COUNT(*) FROM dbo.sw_people;

SELECT TOP 10 * FROM dbo.sw_planets;
SELECT COUNT(*) FROM dbo.sw_planets;
```

If you previously created these tables with incorrect schema/files, you can
delete the old table folders from the Landing Zone and rerun this notebook to
recreate them cleanly.